# `_RUN_.ipynb` — drive **`src_1`**

The first code cell runs **`_my_builder_1/auto.py`** via `runpy` (same `sys.path` setup as `python auto.py`, without running `main()`). Discovery picks **`auto.py` only if** a sibling **`src_1/__init__.py`** exists — so the repo-root `auto.py` (src_2) is never used.

The pickle **`_dta_raw_fetched.pkl`** lives next to the `src_1` package (`default_raw_pkl_path()`), not relative to the notebook’s cwd.

- No pickle → download all **`entries.FETCH_STOCKS`**.
- Pickle OK and has every symbol → **load only**.
- New tickers in `FETCH_STOCKS` → fetch **only missing** and merge.

Edit **`src_1/entries.py`** for `FETCH_STOCKS` and `MY_STOCKS`.

In [1]:
# --- Cell 0: load or incrementally update raw price panel (.pkl) ---
import os
import runpy
import sys
from pathlib import Path


def _bootstrap_src1_via_auto() -> None:
    """Run `auto.py` that sits next to `src_1/` (not repo-root `auto.py` for src_2)."""
    mark = "auto.py"
    here = Path.cwd().resolve()

    def is_src1_builder_auto(p: Path) -> bool:
        return (
            p.is_file()
            and p.name == mark
            and (p.parent / "src_1" / "__init__.py").is_file()
        )

    def try_run(p: Path) -> bool:
        if not is_src1_builder_auto(p):
            return False
        runpy.run_path(str(p), run_name="src1_notebook_bootstrap")
        return True

    for base in [here, *here.parents][:45]:
        if try_run(base / mark):
            return
        try:
            for ch in sorted(base.iterdir()):
                if ch.is_dir() and try_run(ch / mark):
                    return
        except OSError:
            continue

    for dirpath, dirnames, filenames in os.walk(str(here)):
        depth = len(Path(dirpath).relative_to(here).parts)
        if depth > 12:
            dirnames[:] = []
            continue
        if mark in filenames:
            p = Path(dirpath) / mark
            if try_run(p):
                return

    raise FileNotFoundError(
        "Could not find auto.py next to src_1/ (e.g. _my_builder_1/auto.py). "
        "Cd to this repo, then Kernel → Restart and run this cell again."
    )


_bootstrap_src1_via_auto()

from src_1._0_fns_get_data import sync_raw_pkl

pkl_path, df_px = sync_raw_pkl(verbose=True)
print("Using:", pkl_path)
print("shape:", df_px.shape, "| index:", df_px.index.min(), "..", df_px.index.max())
sorted(set(df_px.columns.get_level_values(0)))[:20]

No new data — using imported pickle only: "_dta_raw_fetched.pkl" (19 symbol(s) already present).
Using: /Users/yerik/_apple_lib/_peg_ProgEnvGit/a4yy_STRATEGY/builders/_my_builder_1/src_1/_dta_raw_fetched.pkl
shape: (1827, 150) | index: 2021-04-18 00:00:00 .. 2026-04-18 00:00:00


['AAPL',
 'ACHR',
 'AMD',
 'ANDE',
 'APPL',
 'ASML',
 'AVGO',
 'COIN',
 'CRSP',
 'CURSOR',
 'HOOD',
 'LOVABLE',
 'NFLX',
 'NVDA',
 'OPEN',
 'PLTR',
 'RBLX',
 'ROKU',
 'SHOP',
 'SOBO']

### `entries` (data only)

Compare **fetch list** vs **positions**.

In [2]:
from src_1 import entries

print("FETCH_STOCKS:", entries.FETCH_STOCKS)
print("MY_STOCKS keys:", list(entries.MY_STOCKS.keys()))

FETCH_STOCKS: ['AAPL', 'TREE', 'NVDA', 'AMD', 'SOXX', 'CRSP', 'TSLA', 'ROKU', 'TSM', 'SSNLF', 'ASML', 'AVGO', 'COIN', 'HOOD', 'PLTR', 'ACHR', 'SHOP', 'SOBO', 'RBLX']
MY_STOCKS keys: ['NFLX', 'NVDA', 'AAPL', 'TREE', 'AMD', 'ASML', 'AVGO', 'SOBO']


### `stk_ledg_df` (`_1_fns_history_by_symbol`)

Ledger from **`MY_STOCKS`** (not from the pkl).

In [3]:
from src_1._1_fns_history_by_symbol import stk_ledg_df

raw = stk_ledg_df()
raw.head()

,date,ticker,kind,shares,split_ratio,div_per_share,reinvest,price_per_share,notes
0,2022-04-19,NFLX,BUY,2.142813,NaN,NaN,0,245.00,
1,2022-04-21,NVDA,BUY,1.378993,NaN,NaN,0,217.55,
2,2022-05-01,NFLX,BUY,2.608105,NaN,NaN,0,191.71,
3,2022-05-04,TREE,BUY,2.558257,NaN,NaN,0,78.18,
4,2022-05-05,NFLX,BUY,2.491156,NaN,NaN,0,200.71,


### `hist_summ_pr` (`_1_fns_history_by_symbol`)

One pass: **history** + **per-ticker summary** (uses `df_px` from cell 0).

In [4]:
from src_1._1_fns_history_by_symbol import hist_summ_pr

hist, summ = hist_summ_pr(raw, df_px)
hist.head(), summ

(  ticker       date                  action  shares_after  cash_flow_usd  \
 0   AAPL 2022-05-16  BUY  3.445187 @ 145.13      3.445187    -499.999989   
 1   AAPL 2022-05-18  BUY  1.404613 @ 142.39      4.849800    -200.002845   
 2   AAPL 2022-06-07  BUY  1.685203 @ 148.35      6.535003    -249.999865   
 3   AAPL 2022-06-14  SELL 2.613960 @ 133.00      3.921043     347.656680   
 4   AAPL 2022-08-11   DIV $0.9000/sh (cash)      3.921043       3.528939   
 
    cash_cumulative_usd  
 0          -499.999989  
 1          -700.002834  
 2          -950.002699  
 3          -602.346019  
 4          -598.817081  ,
   ticker  shares_now  close_today  position_value_usd  net_cash_flow_usd  \
 0   AAPL    1.955959   270.230011          528.558822         -76.104966   
 1    AMD    0.460000   278.390015          128.059407        -125.897400   
 2   ASML    0.039000  1459.800049           56.932202         -55.273920   
 3   AVGO    0.056000   406.540009           22.766240         -22.2409

### `sym_hist_tb` (`_1_fns_history_by_symbol`)

History table only.

In [5]:
from src_1._1_fns_history_by_symbol import sym_hist_tb

sym_hist_tb(raw, df_px).head()

,ticker,date,action,shares_after,cash_flow_usd,cash_cumulative_usd
0,AAPL,2022-05-16,BUY 3.445187 @ 145.13,3.445187,-499.999989,-499.999989
1,AAPL,2022-05-18,BUY 1.404613 @ 142.39,4.849800,-200.002845,-700.002834
2,AAPL,2022-06-07,BUY 1.685203 @ 148.35,6.535003,-249.999865,-950.002699
3,AAPL,2022-06-14,SELL 2.613960 @ 133.00,3.921043,347.656680,-602.346019
4,AAPL,2022-08-11,DIV $0.9000/sh (cash),3.921043,3.528939,-598.817081


### `entry_gl_tb` (`_2_gainloss`)

% gain/loss per BUY — walks every symbol in **`MY_STOCKS`**.

In [6]:
from src_1._2_gainloss import entry_gl_tb

entry_gl_tb(hist, df_px)

,ticker,entry_date,adj_entry_price,last_close,pct_change
0,AAPL,2022-05-16,145.130,270.230011,86.198588
1,AAPL,2022-05-18,142.390,270.230011,89.781594
2,AAPL,2022-06-07,148.350,270.230011,82.157068
3,AAPL,2024-05-16,190.220,270.230011,42.061829
4,AAPL,2024-08-15,224.470,270.230011,20.385803
5,AAPL,2024-11-14,224.620,270.230011,20.305410
6,AAPL,2025-02-13,244.060,270.230011,10.722778
7,AAPL,2025-05-15,210.380,270.230011,28.448527
8,AAPL,2025-08-14,232.450,270.230011,16.252962
9,AAPL,2025-11-13,272.490,270.230011,-0.829384


### `portf_sum_tb` (`_3_summary`)

Final **per-ticker** table (last output in `run.py`).

In [7]:
from src_1._3_summary import portf_sum_tb

portf_sum_tb(summ)

,ticker,shares_now,close_today,position_value_usd,net_cash_flow_usd,n_buys,n_sells,n_splits,n_divs
0,AAPL,1.955959,270.230011,528.558822,-76.104966,11,2,0,7
1,AMD,0.460000,278.390015,128.059407,-125.897400,1,0,0,0
2,ASML,0.039000,1459.800049,56.932202,-55.273920,1,0,0,0
3,AVGO,0.056000,406.540009,22.766240,-22.240960,1,0,0,0
4,NFLX,28.148410,97.309998,2739.121708,-553.392299,4,2,1,0
5,NVDA,5.150657,201.679993,1038.784565,-166.642503,2,1,1,16
6,SOBO,0.026674,32.070000,0.855435,-0.709929,6,0,0,6
7,TREE,1.028257,47.959999,49.315205,-115.869832,1,1,0,0
